# 00 – Setup & Erstmigration

Dieses Notebook richtet die Databricks-Umgebung ein und führt die **Erstmigration** durch.

**Einmalig ausführen** – danach für Updates `01_delta_update.ipynb` verwenden.

## Ablauf
1. Secrets prüfen (GitHub Codespaces)
2. Verbindung zu Databricks testen
3. Schema `main.polar` anlegen
4. Alle 5 Delta Tables erstellen
5. Erstes ZIP importieren
6. Tabellenübersicht ausgeben

---
> ⚠️ **Voraussetzung:** Alle 5 GitHub Codespaces Secrets müssen gesetzt sein.  
> Repository → Settings → Secrets and variables → Codespaces

## 1 – Imports & Secrets prüfen

In [5]:
import os
import sys

# src/ zum Python-Pfad hinzufügen
sys.path.insert(0, '/workspaces/polar_databricks/src')

from db_loader import DatabricksLoader, secrets_pruefen

# ── Secrets prüfen ──────────────────────────────────────────
# Wirft EnvironmentError mit Schritt-für-Schritt-Anleitung
# wenn ein Secret fehlt
secrets_pruefen()

✅ Alle Databricks Secrets gefunden.


## 2 – Verbindung zu Databricks testen

In [6]:
# Verbindungsparameter anzeigen (Token wird nicht ausgegeben)
print("Verbindungsparameter:")
print(f"  Host      : {os.environ.get('DATABRICKS_HOST')}")
print(f"  HTTP-Path : {os.environ.get('DATABRICKS_HTTP_PATH')}")
print(f"  Catalog   : {os.environ.get('DATABRICKS_CATALOG', 'main')}")
print(f"  Schema    : {os.environ.get('DATABRICKS_SCHEMA', 'polar')}")
print(f"  Token     : {'*' * 20} (ausgeblendet)")
print()

# Verbindung aufbauen und testen
db = DatabricksLoader()
ok = db.verbindung_testen()

if not ok:
    raise RuntimeError(
        "Verbindungstest fehlgeschlagen!\n"
        "→ SQL Warehouse im Databricks UI starten\n"
        "→ Token prüfen: Databricks UI → User Settings → Developer"
    )

Verbindungsparameter:
  Host      : https://dbc-440507f2-1163.cloud.databricks.com
  HTTP-Path : /sql/1.0/warehouses/f9476e8729ec9452
  Catalog   : workspace
  Schema    : polar
  Token     : ******************** (ausgeblendet)

✅ Alle Databricks Secrets gefunden.
🔧 DatabricksLoader initialisiert
   Catalog : workspace
   Schema  : polar
   Host    : https://dbc-440507f2-1163.cloud.databricks.com
✅ Databricks verbunden
✅ Datenbankverbindung funktioniert


## 3 – Schema anlegen

In [7]:
catalog = os.environ.get('DATABRICKS_CATALOG', 'main')
schema  = os.environ.get('DATABRICKS_SCHEMA', 'polar')

print(f"Lege Schema an: {catalog}.{schema}")

db.abfrage(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

# Schema-Existenz bestätigen
df_schemas = db.abfrage(
    f"SHOW SCHEMAS IN {catalog} LIKE '{schema}'"
)
if not df_schemas.empty:
    print(f"✅ Schema '{catalog}.{schema}' ist bereit")
else:
    print(f"⚠️  Schema konnte nicht verifiziert werden – trotzdem fortfahren")

Lege Schema an: workspace.polar
✅ Schema 'workspace.polar' ist bereit


## 4 – Delta Tables erstellen

Alle Tabellen werden mit `CREATE TABLE IF NOT EXISTS` angelegt –
bestehende Daten bleiben erhalten.

In [8]:
# ── Tabellen-Definitionen ────────────────────────────────────
tabellen_sql = {

    'activity': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.activity (
            datum             DATE    NOT NULL COMMENT 'Datum des Aktivitätstages',
            schritte          BIGINT          COMMENT 'Tägliche Schrittanzahl',
            kalorien          DOUBLE          COMMENT 'Verbrauchte Kalorien (kcal)',
            schlaf_stunden    DOUBLE          COMMENT 'Schlafdauer in Stunden',
            schlaf_qualitaet  DOUBLE          COMMENT 'Schlafqualität 0-1',
            met_minuten       DOUBLE          COMMENT 'Metabolische Äquivalent-Minuten'
        )
        USING DELTA
        COMMENT 'Tägliche Aktivitätsdaten aus Polar activity_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'training': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.training (
            datum        DATE    NOT NULL COMMENT 'Datum der Trainingseinheit',
            sport        STRING  NOT NULL COMMENT 'Sportart (z.B. RUNNING, CYCLING)',
            dauer_min    DOUBLE           COMMENT 'Trainingsdauer in Minuten',
            hr_avg       DOUBLE           COMMENT 'Durchschnittliche Herzfrequenz (bpm)',
            hr_max       DOUBLE           COMMENT 'Maximale Herzfrequenz (bpm)',
            distanz_km   DOUBLE           COMMENT 'Zurückgelegte Distanz in km',
            kalorien     DOUBLE           COMMENT 'Verbrannte Kalorien (kcal)',
            wochentag    STRING           COMMENT 'Wochentag (Monday..Sunday)',
            jahr         INT              COMMENT 'Jahr der Trainingseinheit'
        )
        USING DELTA
        COMMENT 'Trainingseinheiten aus Polar training_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'heartrate': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.heartrate (
            datum         DATE    NOT NULL COMMENT 'Messdatum',
            hr_ruhepuls   DOUBLE           COMMENT 'Ruhepuls: 5. Perzentil aller Tages-HR-Werte',
            hr_mean       DOUBLE           COMMENT 'Mittlere Herzfrequenz des Tages',
            hr_max        INT              COMMENT 'Maximale Herzfrequenz des Tages',
            wochentag_nr  INT              COMMENT 'Wochentag 0=Montag, 6=Sonntag',
            monat         INT              COMMENT 'Monat 1-12'
        )
        USING DELTA
        COMMENT 'Tägliche HR-Aggregate aus Polar 247ohr_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'hrv': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.hrv (
            datum           DATE    NOT NULL COMMENT 'Messdatum',
            hrv_rmssd       DOUBLE           COMMENT 'Root Mean Square Successive Differences (ms)',
            hrv_sdnn        DOUBLE           COMMENT 'Standard Deviation NN Intervals (ms)',
            ppi_mean_ms     DOUBLE           COMMENT 'Mittleres Peak-to-Peak Intervall (ms)',
            hr_aus_ppi      DOUBLE           COMMENT 'Herzfrequenz abgeleitet aus PPI (bpm)',
            anzahl_samples  BIGINT           COMMENT 'Anzahl PPI-Messwerte des Tages'
        )
        USING DELTA
        COMMENT 'HRV-Metriken aus Polar ppi_*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'fitness_tests': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.fitness_tests (
            datum               DATE    NOT NULL COMMENT 'Datum des Fitness-Tests',
            own_index           DOUBLE           COMMENT 'Polar OwnIndex (Fitness-Score)',
            avg_hr              DOUBLE           COMMENT 'Durchschnittliche Herzfrequenz während Test (bpm)',
            fitness_class       STRING           COMMENT 'Fitness-Klasse (z.B. ELITE, GOOD)',
            timezone_offset     INT              COMMENT 'Zeitzonen-Offset in Minuten',
            birthday            DATE             COMMENT 'Geburtsdatum',
            sex                 STRING           COMMENT 'Geschlecht (MALE, FEMALE)',
            height              DOUBLE           COMMENT 'Größe in cm',
            weight              DOUBLE           COMMENT 'Gewicht in kg',
            max_hr              INT              COMMENT 'Maximale Herzfrequenz (bpm)',
            resting_hr          INT              COMMENT 'Ruheherzfrequenz (bpm)',
            aerobic_threshold   INT              COMMENT 'Aerobe Schwelle (bpm)',
            anaerobic_threshold INT              COMMENT 'Anaerobe Schwelle (bpm)',
            vo2max              DOUBLE           COMMENT 'VO2Max in ml/kg/min',
            training_background STRING           COMMENT 'Trainingshintergrund',
            weight_source       STRING           COMMENT 'Gewichtsquelle',
            sleep_goal          BIGINT           COMMENT 'Schlafziel in Millisekunden'
        )
        USING DELTA
        COMMENT 'Fitness-Test-Ergebnisse aus Polar fitness-test-results-*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'orthostatic_tests': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.orthostatic_tests (
            datum     DATE   NOT NULL COMMENT 'Datum des Tests',
            json_data STRING          COMMENT 'Vollständige JSON-Daten als String'
        )
        USING DELTA
        COMMENT 'Orthostatische Tests aus Polar orthostatic-test-result-*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'physical_info': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.physical_info (
            datum     DATE   NOT NULL COMMENT 'Datum der Information',
            json_data STRING          COMMENT 'Vollständige JSON-Daten als String'
        )
        USING DELTA
        COMMENT 'Physische Informationen aus Polar physical-info-*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'products_devices': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.products_devices (
            datum     DATE   NOT NULL COMMENT 'Datum der Registrierung',
            json_data STRING          COMMENT 'Vollständige JSON-Daten als String'
        )
        USING DELTA
        COMMENT 'Geräte- und Produktinformationen aus Polar products-devices-*.json'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'sonstige_daten': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.sonstige_daten (
            dateiname STRING NOT NULL COMMENT 'Originaler Dateiname',
            kategorie STRING NOT NULL COMMENT 'Dateikategorie',
            datum     DATE            COMMENT 'Extrahiertes Datum (falls verfügbar)',
            json_data STRING          COMMENT 'Vollständige JSON-Daten als String'
        )
        USING DELTA
        COMMENT 'Sonstige Polar-Daten (generische Speicherung)'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,

    'import_log': f"""
        CREATE TABLE IF NOT EXISTS {catalog}.{schema}.import_log (
            dateiname     STRING    NOT NULL COMMENT 'Originaler Dateiname im ZIP',
            hash_md5      STRING    NOT NULL COMMENT 'MD5-Hash des Dateiinhalts',
            importiert_am TIMESTAMP          COMMENT 'Zeitpunkt des Imports',
            kategorie     STRING             COMMENT 'Dateikategorie'
        )
        USING DELTA
        COMMENT 'Import-Protokoll – verhindert Doppelimport'
        TBLPROPERTIES ('delta.autoOptimize.optimizeWrite' = 'true')
    """,
}

# ── Tabellen anlegen ─────────────────────────────────────────
for name, sql in tabellen_sql.items():
    try:
        db.abfrage(sql)
        print(f"✅ {catalog}.{schema}.{name} – bereit")
    except Exception as e:
        print(f"❌ Fehler bei '{name}': {e}")

✅ workspace.polar.activity – bereit
✅ workspace.polar.training – bereit
✅ workspace.polar.heartrate – bereit
✅ workspace.polar.hrv – bereit
✅ workspace.polar.fitness_tests – bereit
✅ workspace.polar.orthostatic_tests – bereit
✅ workspace.polar.physical_info – bereit
✅ workspace.polar.products_devices – bereit
✅ workspace.polar.sonstige_daten – bereit
✅ workspace.polar.import_log – bereit


## 5 – Erstimport: ZIP verarbeiten

ZIP-Datei in `input/` ablegen, dann diese Zelle ausführen.

In [9]:
import glob
from pathlib import Path

# Verfügbare ZIP-Dateien anzeigen
input_ordner = Path('..') / 'input'
zip_dateien  = sorted(input_ordner.glob('*.zip'))

print(f"ZIP-Dateien in '{input_ordner}':")
if not zip_dateien:
    print("   ⚠️  Keine ZIP-Dateien gefunden!")
    print("   → Polar-Export-ZIP in den 'input/' Ordner kopieren")
    print("   → Polar App: Profil → Einstellungen → Eigene Daten exportieren")
else:
    for z in zip_dateien:
        groesse_mb = z.stat().st_size / 1024 / 1024
        print(f"   📦 {z.name}  ({groesse_mb:.1f} MB)")

ZIP-Dateien in '../input':
   ⚠️  Keine ZIP-Dateien gefunden!
   → Polar-Export-ZIP in den 'input/' Ordner kopieren
   → Polar App: Profil → Einstellungen → Eigene Daten exportieren


In [10]:
# ── ZIP importieren ──────────────────────────────────────────
# Diese Zelle nur ausführen wenn zip_dateien nicht leer ist!

if not zip_dateien:
    print("⏭️  Kein ZIP vorhanden – Zelle übersprungen")
else:
    from delta_updater import DeltaUpdater

    updater = DeltaUpdater()

    # Alle ZIP-Dateien verarbeiten (normalerweise nur eine)
    for zip_pfad in zip_dateien:
        print(f"\n▶ Verarbeite: {zip_pfad.name}")
        bericht = updater.verarbeite_zip(str(zip_pfad))

    print("\n🎉 Erstimport abgeschlossen!")

⏭️  Kein ZIP vorhanden – Zelle übersprungen


## 6 – Tabellenübersicht & Validierung

In [ ]:
# Übersicht aller Tabellen
df_uebersicht = db.tabellen_uebersicht()


📊 Tabellen-Übersicht:
  tabelle  zeilen  min_datum  max_datum
 activity    3751 2014-04-25 2026-03-15
 training    3125 2014-03-24 2026-03-13
heartrate    1710 2018-08-31 2026-03-15
      hrv     369 2024-12-23 2026-03-15


In [ ]:
# Erste Zeilen jeder Tabelle zur Validierung
print("\n── Activity (erste 3 Zeilen) ──")
display(db.lade_activity().head(3))

print("\n── Training (erste 3 Zeilen) ──")
display(db.lade_training().head(3))

print("\n── Herzfrequenz (erste 3 Zeilen) ──")
display(db.lade_heartrate().head(3))

print("\n── HRV (erste 3 Zeilen) ──")
display(db.lade_hrv().head(3))


── Activity (erste 3 Zeilen) ──
✅ Activity geladen: 3751 Zeilen


,datum,schritte,kalorien,schlaf_stunden,schlaf_qualitaet,met_minuten
0,2014-04-25,0.0,1552.0,NaN,0.0,0.0
1,2014-04-26,0.0,1546.0,NaN,0.0,0.0
2,2014-04-27,0.0,1547.0,NaN,0.0,0.0



── Training (erste 3 Zeilen) ──
✅ Training geladen: 3125 Einheiten


,datum,sport,dauer_min,hr_avg,hr_max,distanz_km,kalorien,wochentag,jahr
0,2014-03-24,MOUNTAIN_BIKING,30.0,NaN,NaN,NaN,135.0,Monday,2014
1,2014-03-24,CYCLING,25.0,NaN,NaN,NaN,39.0,Monday,2014
2,2014-03-25,CYCLING,25.0,NaN,NaN,NaN,39.0,Tuesday,2014



── Herzfrequenz (erste 3 Zeilen) ──
✅ Herzfrequenz geladen: 1710 Tage


,datum,hr_ruhepuls,hr_mean,hr_max,wochentag_nr,monat
0,2018-08-31,60.5,80.3,111,4,8
1,2018-09-01,64.0,77.3,118,5,9
2,2018-09-02,60.0,78.0,117,6,9



── HRV (erste 3 Zeilen) ──
✅ HRV geladen: 369 Tage


,datum,hrv_rmssd,hrv_sdnn,ppi_mean_ms,hr_aus_ppi,anzahl_samples
0,2024-12-23,94.62,126.37,759.9,79.0,101696
1,2024-12-25,54.03,108.21,827.2,72.5,50009
2,2024-12-26,85.92,175.89,803.8,74.6,87636


In [ ]:
# Import-Log prüfen
print("── Import-Log Übersicht ──")
display(db.import_log_uebersicht())

print("\n✅ Setup abgeschlossen – weiter mit 01_delta_update.ipynb")
db.schliessen()

── Import-Log Übersicht ──
✅ Import-Log: 0 Kategorien


,kategorie,anzahl,letzter_import



✅ Setup abgeschlossen – weiter mit 01_delta_update.ipynb
✅ Datenbankverbindung geschlossen
